# E8: Broad-sense heritability of PhenoSuite-extracted traits

**Question.** Does the PhenoSuite representation preserve heritable variation? Specifically:
1. Does PhenoSuite's spline-based trait extractor preserve the heritability of the gold-standard voxel-PCA pipeline? (override descriptor: E1b)
2. Does the 3-parameter procedural compression preserve heritability further? (procedural descriptor: E5)

**Method.** Compute broad-sense heritability H² on θ and φ-derivative traits using Davis et al. 2025's exact lme4 random-intercept model (`trait ~ 1 + (1|PI_num)`, n≥ 2 per genotype, reference genotype PI_656058 excluded). Report H² with both the simple form σ²_G / (σ²_G + σ²_e) and the Davis-paper Eq. 3 form σ²_G / (σ²_G + σ²_e / 2).

**Three trait sources, same plants, same lme4 model:**
- **gold** — voxel-PCA θ/φ from `angles.txt` (published method, Gaillard 2021).
- **E1b** — PhenoSuite's spline-based trait extractor on the override descriptor (raw skeleton, no compression).
- **E5** — PhenoSuite's trait extractor on the 3-parameter procedural fit (`leaf_angle`, `droopiness`, `leaf_length`).

**Published comparison point.** Davis et al. 2025 report that 13 of 25 phyllotaxy summary metrics have H² ≥ 0.20 (their per-trait values are in Supplementary Table 1). We use H² ≥ 0.20 as the published threshold for 'heritable'.

In [ ]:
from __future__ import annotations

import re
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

REPO_ROOT = Path.cwd().parents[1] if Path.cwd().name == '01_phyllotaxis' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from experiments import data_loaders as dl
from experiments.paths import assert_data_present

assert_data_present()

OUT = REPO_ROOT / 'experiments' / '01_phyllotaxis' / 'outputs'
FIG = REPO_ROOT / 'experiments' / '01_phyllotaxis' / 'figures'
FIG.mkdir(parents=True, exist_ok=True)

REFERENCE_GENOTYPE = 'PI656058'  # excluded per Davis 2025 calcHeritability
LOWER_LEAVES = list(range(5))    # lower 5 leaves per Davis convention
MIN_REPS = 2                     # n>=2 plants per genotype

print('Imports OK.')

## 1. Load and merge the three trait sources

Both `e1b_per_leaf_phenosuite_vs_gold.csv` and `e5_roundtrip_vs_gold.csv` carry the gold-standard `theta_gold` / `phi_gold` for the same skeletons. We use E1b's gold column as canonical and merge in E1b's `theta_ours/phi_ours` (override extractor) and E5's `theta_rt/phi_rt` (procedural extractor).

In [ ]:
e1b = pd.read_csv(OUT / 'e1b_per_leaf_phenosuite_vs_gold.csv')
e5  = pd.read_csv(OUT / 'e5_roundtrip_vs_gold.csv')

leaf = (
    e1b[['plant_id', 'leaf_index', 'theta_gold', 'phi_gold',
         'theta_ours', 'phi_ours', 'leaf_voxel_count']]
    .rename(columns={'theta_ours': 'theta_e1b', 'phi_ours': 'phi_e1b'})
    .merge(
        e5[['plant_id', 'leaf_index', 'theta_rt', 'phi_rt']]
          .rename(columns={'theta_rt': 'theta_e5', 'phi_rt': 'phi_e5'}),
        on=['plant_id', 'leaf_index'], how='inner',
    )
)
print(f'Merged per-leaf rows: {len(leaf):,}')
print(f'Plants:               {leaf["plant_id"].nunique()}')
leaf.head(3)

## 2. Map plants to genotypes

Plant directory names embed a JS line ID (e.g. `JS39-65`). We extract the JS prefix, normalize to `JS<n>`, and look up the PI accession via Davis's `JSname_PIname.csv`. Plants with no mapping or the reference genotype are dropped.

In [ ]:
def norm_js(s: str | None) -> str | None:
    if not isinstance(s, str):
        return None
    m = re.match(r'(?i)js(\d+)', s)
    return f'JS{m.group(1)}' if m else None

geno, js_pi = dl.load_jensina_genotypes()
js_pi['JS_norm'] = js_pi['JS_ID'].apply(norm_js)
js_to_pi = dict(zip(js_pi['JS_norm'], js_pi['PI#'].str.replace('_', '').str.upper()))

leaf['js_norm'] = leaf['plant_id'].apply(
    lambda pid: norm_js(dl.parse_plant_id(pid)['js_id'])
)
leaf['PI_num'] = leaf['js_norm'].map(js_to_pi)

n_total = leaf['plant_id'].nunique()
n_mapped = leaf.loc[leaf['PI_num'].notna(), 'plant_id'].nunique()
n_in_geno = leaf.loc[leaf['PI_num'].isin(set(geno['x'])), 'plant_id'].nunique()
print(f'Plants:                 {n_total}')
print(f'  with PI mapping:       {n_mapped}')
print(f'  in Davis genotypes.csv:{n_in_geno}')

# Drop reference genotype and unmapped plants
leaf = leaf[leaf['PI_num'].notna() & (leaf['PI_num'] != REFERENCE_GENOTYPE)].copy()
print(f'After reference + null drop: {leaf["plant_id"].nunique()} plants, {leaf["PI_num"].nunique()} genotypes')

## 3. Restrict to lower 5 leaves and to genotypes with n≥2 plants

Davis's heritability analysis is conducted on the lower 5 phyllotaxic angles — the upper canopy is too noisy because of plant movement during rotation imaging. Heritability requires within-genotype variance, so any singleton genotype contributes nothing and must be dropped.

In [ ]:
leaf = leaf[leaf['leaf_index'].isin(LOWER_LEAVES)].copy()

# Count plants per genotype and filter
plant_pi = leaf.drop_duplicates('plant_id')[['plant_id', 'PI_num']]
counts = plant_pi.groupby('PI_num').size()
rep_pi = set(counts[counts >= MIN_REPS].index)
leaf_rep = leaf[leaf['PI_num'].isin(rep_pi)].copy()

n_plants = leaf_rep['plant_id'].nunique()
n_geno = leaf_rep['PI_num'].nunique()
rep_counts = plant_pi[plant_pi['PI_num'].isin(rep_pi)].groupby('PI_num').size()
print(f'Replicated subset: {n_plants} plants, {n_geno} genotypes')
print(f'  reps/geno  min={rep_counts.min()}  median={int(rep_counts.median())}  max={rep_counts.max()}')
print(f'  distribution: {dict(rep_counts.value_counts().sort_index())}')

## 4. Davis-convention features

Two transformations from the R code:

- `normalize360(x) = x mod 360` — wrap to `[0, 360)`.
- `normalize180(x) = |normalize360(x) - 180|` — deviation from 180° in `[0, 180]`.

Raw φ is just a stem rotation; the heritable phenotype is the *consecutive-leaf phyllotaxic deviation* from 180°:

$$\Phi_i = \mathrm{normalize180}(\varphi_{i+1} - \varphi_i)$$

We compute this for i = 0..3 (giving Φ₁..Φ₄) and aggregate per plant via median (`med_phi_dev`) and arithmetic mean (`am_phi_dev`). For θ we use the raw per-node values and their per-plant median (`med_theta`) and mean (`am_theta`).

In [ ]:
def norm360(x):
    return np.mod(x, 360.0)

def norm180(x):
    return np.abs(norm360(x) - 180.0)

SRC = ['gold', 'e1b', 'e5']

# Normalize phi to [0, 360)
for s in SRC:
    leaf_rep[f'phi_{s}'] = norm360(leaf_rep[f'phi_{s}'])

# Pivot to wide: one row per plant
wide = (
    leaf_rep
      .pivot_table(
          index=['plant_id', 'PI_num'],
          columns='leaf_index',
          values=[f'phi_{s}' for s in SRC] + [f'theta_{s}' for s in SRC],
          aggfunc='first',
      )
)
wide.columns = [f'{a}_{b}' for a, b in wide.columns]
wide = wide.reset_index()

# Consecutive-leaf deviations Phi_i = |normalize360(phi_{i+1} - phi_i) - 180|  (i=1..4)
for s in SRC:
    dev_cols = []
    for i in range(1, 5):
        c = f'Phi_dev_{s}_{i}'
        wide[c] = norm180(wide[f'phi_{s}_{i}'] - wide[f'phi_{s}_{i-1}'])
        dev_cols.append(c)
    wide[f'med_phi_dev_{s}'] = wide[dev_cols].median(axis=1, skipna=True)
    wide[f'am_phi_dev_{s}']  = wide[dev_cols].mean(axis=1, skipna=True)
    theta_cols = [f'theta_{s}_{i}' for i in LOWER_LEAVES]
    wide[f'med_theta_{s}'] = wide[theta_cols].median(axis=1, skipna=True)
    wide[f'am_theta_{s}']  = wide[theta_cols].mean(axis=1, skipna=True)

print(f'Per-plant rows: {len(wide)}')
wide.head(2)

## 5. Davis's lme4 model in Python

`calcHeritability` from `PhyllotaxyAnalysis.R`:

```r
fit  <- lmer(trait ~ (1|PI_num), data = dfh)   # filter to n>1
sigma <- as.numeric(VarCorr(fit)$Std.Dev.)
h    <- sigma[1]^2 / (sigma[1]^2 + sigma[2]^2)             # calcHeritability
h2   <- sigma[1]^2 / (sigma[1]^2 + (sigma[2]^2) / 2)       # calcHeritability2 (paper Eq. 3)
```

We use `statsmodels.MixedLM(trait ~ 1, groups=PI_num)` with REML, which is the Python equivalent of lme4's `lmer(... (1|PI_num))`. Where REML pins σ²_G to the boundary at 0 (a real possibility when between-genotype variance is small), we report 0 — same convention as Davis.

In [ ]:
def calc_heritability(df: pd.DataFrame, trait: str, group: str = 'PI_num') -> dict:
    """Davis-style broad-sense heritability via lme4-equivalent REML mixed model.

    Returns dict with sigma2_G, sigma2_e, H2 (unadjusted), H2_n2 (Davis Eq. 3 with n=2),
    n_plants and n_genotypes.

    Implementation note: statsmodels MixedLM with method='lbfgs' is known to get stuck
    at the sigma_G=0 boundary (returns 0 even when ANOVA method-of-moments shows real
    between-genotype variance). We try `bfgs` first, then `powell`, then `lbfgs` as a
    last resort. This matches the R `lmer` solution to within numerical precision on
    the test traits we cross-checked.
    """
    sub = df[[group, trait]].dropna()
    sub = sub.rename(columns={trait: 'y'})
    # Enforce n>1 per genotype (matches Davis's `filter(n>1)`)
    n_per = sub.groupby(group).size()
    sub = sub[sub[group].isin(n_per[n_per > 1].index)].copy()
    n_plants = len(sub)
    n_geno = sub[group].nunique()
    out = {
        'trait': trait, 'n_plants': n_plants, 'n_genotypes': n_geno,
        'sigma2_G': np.nan, 'sigma2_e': np.nan,
        'H2': np.nan, 'H2_n2': np.nan,
    }
    if n_geno < 2:
        return out
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        sg2 = se2 = None
        for method in ('bfgs', 'powell', 'lbfgs'):
            try:
                md = smf.mixedlm('y ~ 1', sub, groups=sub[group])
                mf = md.fit(method=method, reml=True)
                if mf.converged:
                    sg2 = float(mf.cov_re.iloc[0, 0])
                    se2 = float(mf.scale)
                    break
            except Exception:
                continue
        if sg2 is None:
            return out
    out['sigma2_G'] = sg2
    out['sigma2_e'] = se2
    denom = sg2 + se2
    denom_n2 = sg2 + se2 / 2.0
    out['H2']    = sg2 / denom    if denom    > 0 else 0.0
    out['H2_n2'] = sg2 / denom_n2 if denom_n2 > 0 else 0.0
    return out

# Smoke test against a trait we already cross-checked with ANOVA MoM (H2_n2 ~ 0.60)
test = calc_heritability(wide, 'med_theta_e5')
print('Smoke test (med_theta_e5):', test)


## 6. Run on all three sources and all traits

Trait set:
- **Per-node θ**: `theta_<src>_0`, ..., `theta_<src>_4` (5 nodes)
- **Per-node Φ** (deviation from alternating): `Phi_dev_<src>_1`, ..., `Phi_dev_<src>_4` (4 differences)
- **Per-plant aggregates**: `med_theta_<src>`, `am_theta_<src>`, `med_phi_dev_<src>`, `am_phi_dev_<src>`

= 5 + 4 + 4 = **13 traits per source**, **39 fits total**.

In [ ]:
def trait_list_for(src: str) -> list[tuple[str, str, str]]:
    """Return [(category, label, column_name), ...] for one source."""
    out = []
    for i in LOWER_LEAVES:
        out.append(('per-node theta', f'theta_{i}', f'theta_{src}_{i}'))
    for i in range(1, 5):
        out.append(('per-node Phi_dev', f'Phi_dev_{i}', f'Phi_dev_{src}_{i}'))
    out.append(('per-plant theta', 'med_theta', f'med_theta_{src}'))
    out.append(('per-plant theta', 'am_theta',  f'am_theta_{src}'))
    out.append(('per-plant Phi_dev', 'med_phi_dev', f'med_phi_dev_{src}'))
    out.append(('per-plant Phi_dev', 'am_phi_dev',  f'am_phi_dev_{src}'))
    return out

rows = []
for src in SRC:
    for cat, label, col in trait_list_for(src):
        res = calc_heritability(wide, col)
        res['source'] = src
        res['category'] = cat
        res['label'] = label
        rows.append(res)

results = pd.DataFrame(rows)
results = results[['source', 'category', 'label', 'trait',
                   'n_plants', 'n_genotypes',
                   'sigma2_G', 'sigma2_e', 'H2', 'H2_n2']]
pd.set_option('display.float_format', '{:.4f}'.format)
results

## 7. Side-by-side H² table (the headline)

Pivot to one row per trait label, one column per source.

In [ ]:
wide_h2 = (
    results.pivot_table(
        index=['category', 'label'],
        columns='source',
        values='H2_n2',
        aggfunc='first',
    )[['gold', 'e1b', 'e5']]
    .reset_index()
)

# Add diagnostic columns: how much H2 is lost going extractor -> compression
wide_h2['extractor_loss']   = wide_h2['gold'] - wide_h2['e1b']  # gold -> E1b
wide_h2['compression_loss'] = wide_h2['e1b']  - wide_h2['e5']   # E1b  -> E5
wide_h2

## 8. Per-trait grouped bar chart

Three bars per trait (gold, E1b, E5). Dashed line at H² = 0.20 = Davis's published threshold for 'heritable'.

In [ ]:
# Paper-quality math labels for the x-axis tick names
def nice_label(raw):
    if raw.startswith('theta_'):
        i = raw.split('_')[1]
        return r'$\theta_{' + i + '}$'
    if raw.startswith('Phi_dev_'):
        i = raw.split('_')[2]
        return r'$\Phi_{' + i + '}$'
    aggregates = {
        'med_theta':   r'median $\theta$',
        'am_theta':    r'mean $\theta$',
        'med_phi_dev': r'median $\Phi$',
        'am_phi_dev':  r'mean $\Phi$',
    }
    return aggregates.get(raw, raw)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
panels = [
    ('per-node theta',   r'Per-node leaf angle $\theta_i$'),
    ('per-node Phi_dev', r'Per-node phyllotaxic deviation $\Phi_i$'),
    ('per-plant',        'Per-plant aggregates'),
]
for ax, (cat_match, title) in zip(axes, panels):
    sub = wide_h2[wide_h2['category'].str.startswith(cat_match)].copy().reset_index(drop=True)
    labels = [nice_label(l) for l in sub['label'].tolist()]
    x = np.arange(len(labels))
    width = 0.27
    ax.bar(x - width, sub['gold'].values, width, label='Voxel',
           color='#999999', edgecolor='black', linewidth=0.5)
    ax.bar(x,         sub['e1b'].values,  width, label='Spline',
           color='#4477aa', edgecolor='black', linewidth=0.5)
    ax.bar(x + width, sub['e5'].values,   width, label='Procedural',
           color='#cc6677', edgecolor='black', linewidth=0.5)
    ax.axhline(0.20, color='black', linestyle='--', linewidth=0.8, alpha=0.6,
               label=r'Davis 2025 $H^2=0.20$ threshold' if ax is axes[0] else None)
    ax.set_xticks(x)
    if cat_match == 'per-plant':
        ax.set_xticklabels(labels, rotation=20, ha='right', fontsize=11)
    else:
        ax.set_xticklabels(labels, rotation=0, fontsize=13)
    ax.tick_params(axis='x', pad=2)
    ax.set_ylabel(r'Broad-sense heritability $H^2$  (Davis Eq. 3, $n=2$)')
    ax.set_title(title, fontsize=11)
    ax.set_ylim(0, 1.0)
    ax.grid(axis='y', alpha=0.3)
    if ax is axes[0]:
        ax.legend(loc='upper right', fontsize=9)
fig.suptitle(f'Heritability across pipelines  (n={n_plants} plants, {n_geno} genotypes, lower {len(LOWER_LEAVES)} leaves)',
             y=1.02, fontsize=12)
fig.tight_layout()
fig.savefig(FIG / 'e8_h2_by_trait.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Scatter: PhenoSuite H² vs gold-standard H²

Points on the y=x line mean perfect heritability preservation. Below the line = signal loss.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True, sharex=True)

for ax, (col, label, color) in zip(axes, [
    ('e1b', 'E1b (override + extractor)', '#4477aa'),
    ('e5',  'E5  (procedural + extractor)', '#cc6677'),
]):
    ax.plot([0, 1], [0, 1], '--', color='black', alpha=0.4, linewidth=1)
    ax.axhline(0.20, color='gray', linestyle=':', linewidth=0.8, alpha=0.7)
    ax.axvline(0.20, color='gray', linestyle=':', linewidth=0.8, alpha=0.7)
    for _, r in wide_h2.iterrows():
        ax.scatter(r['gold'], r[col], s=40, color=color, edgecolor='black',
                   linewidth=0.5, alpha=0.8)
        ax.annotate(r['label'], (r['gold'], r[col]),
                    fontsize=7, xytext=(3, 3), textcoords='offset points')
    ax.set_xlim(-0.02, 1.0)
    ax.set_ylim(-0.02, 1.0)
    ax.set_xlabel('Gold-standard H^2 (voxel PCA)')
    ax.set_ylabel(f'{label} H^2')
    ax.set_title(label)
    ax.grid(alpha=0.3)

fig.suptitle('Heritability preservation: PhenoSuite vs published voxel-PCA pipeline',
             y=1.02, fontsize=12)
fig.tight_layout()
fig.savefig(FIG / 'e8_h2_vs_gold_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Summary counts and save

In [ ]:
summary = {}
for src in SRC:
    sub = results[results['source'] == src]
    summary[src] = {
        'n_traits': len(sub),
        'n_traits_H2_ge_0.20':   int((sub['H2_n2'] >= 0.20).sum()),
        'n_traits_H2_ge_0.40':   int((sub['H2_n2'] >= 0.40).sum()),
        'median_H2_n2':          float(sub['H2_n2'].median()),
        'mean_H2_n2':            float(sub['H2_n2'].mean()),
        'max_H2_n2':             float(sub['H2_n2'].max()),
        'top_trait':             str(sub.loc[sub['H2_n2'].idxmax(), 'trait']),
    }
summary_df = pd.DataFrame(summary).T.reset_index().rename(columns={'index': 'source'})
summary_df

In [ ]:
results.to_csv(OUT / 'e8_heritability.csv', index=False)
wide_h2.to_csv(OUT / 'e8_h2_by_trait.csv', index=False)
summary_df.to_csv(OUT / 'e8_summary.csv', index=False)
print('Wrote outputs:')
for f in ['e8_heritability.csv', 'e8_h2_by_trait.csv', 'e8_summary.csv']:
    print(f'  {OUT / f}')
print('\nWrote figures:')
for f in ['e8_h2_by_trait.png', 'e8_h2_vs_gold_scatter.png']:
    print(f'  {FIG / f}')

## Discussion

**Setup.** Davis et al. 2025 lme4 random-intercept model on the lower 5 leaves of {n_plants} plants spanning {n_geno} genotypes (with ≥2 biological replicates each). Reference genotype `PI_656058` excluded. H² reported with the n=2 adjustment that appears in Davis paper Eq. 3.

**Reading the table.** Three sources sit between two extremes:
- **gold** — the published voxel-PCA pipeline applied directly to the skeleton. Sets the upper bound on extractable heritable signal.
- **E1b** — PhenoSuite's chord-based trait math applied to a full-spline reconstruction of the raw skeleton. Any H² gap relative to gold is *extractor* loss.
- **E5** — PhenoSuite's trait math applied to the 3-parameter procedural fit. Any H² gap relative to E1b is *compression* loss (the part attributable to forcing the skeleton through `leaf_angle`, `droopiness`, `leaf_length`).

**What the H² ≥ 0.20 threshold means.** Davis 2025 reports that 13 of 25 phyllotaxy summary metrics meet this bar; it is the published 'heritable' threshold for this trait family. We count traits that clear it for each pipeline in `e8_summary.csv`.

**Caveats.**
- This biological-replicate design (different physical plants of the same genotype) is more conservative than Davis's timepoint-replicate design (same plant on different days), so absolute H² magnitudes may differ from the paper's numbers. The within-pipeline comparisons (gold vs E1b vs E5) are not affected.
- N=44 genotypes is small; H² standard errors are wide and small absolute differences between sources are not interpretable.
- Upper-canopy heritability is expected to be near zero (Davis: "heritability of measurements from the 3D reconstructions decreased at higher leaves due to movement of the upper canopy during rotation in the imaging"). Per-node `theta_3`/`theta_4` and `Phi_dev_4` numbers are reported but should be interpreted with that limitation in mind.

**Outputs:** `e8_heritability.csv` (long form, all variance components), `e8_h2_by_trait.csv` (wide form, gold/e1b/e5 side-by-side), `e8_summary.csv` (per-source counts). Figures: `e8_h2_by_trait.png`, `e8_h2_vs_gold_scatter.png`.